# Day 24 — Evaluation Metrics Guide

This guide selects and justifies machine learning evaluation metrics for four distinct business scenarios: Fraud Detection, Customer Churn, Medical Diagnosis, and Dynamic Pricing. It also analyzes key edge/error cases that models encounter in production.

**🎯 Goal:** Understand how to align technical metrics with business goals and error costs.

## Scenario 1: Fraud Detection

### Metric Selected: Precision-Recall Area Under Curve (PR-AUC) and Recall
* **Primary Metric:** Recall
* **Ranking/Aggregated Metric:** PR-AUC

### Justification:
* **Business Context:** Fraud is extremely rare (high class imbalance, e.g. < 0.1% of transactions are fraudulent).
* **Cost of Errors:**
  * **False Negative (FN):** Missing a fraudulent transaction leads to direct financial loss, chargeback fees, and reputational damage. The cost of a False Negative is extremely high.
  * **False Positive (FP):** Flagging a legitimate transaction as fraud causes minor customer friction (requiring verification or a quick security call). The cost of a False Positive is low.
* **Why not Accuracy?** Accuracy is misleading. If 99.9% of transactions are legitimate, a model predicting "never fraud" achieves 99.9% accuracy but catches 0% of fraud.
* **Why not ROC-AUC?** ROC-AUC uses False Positive Rate (`FP / (TN + FP)`). Because the number of True Negatives (`TN`) is massive, the denominator is huge, which keeps the FPR artificially close to 0. This makes ROC-AUC look overly optimistic. PR-AUC ignores `TN` and focuses on the minority class (`Recall` and `Precision`), providing a true assessment of model performance.

## Scenario 2: Customer Churn

### Metric Selected: F1-Score (or custom Cost-Benefit Weighted Utility)
* **Primary Metric:** F1-Score
* **Optimized Metric:** F-beta score (with beta adjusted for retention cost)

### Justification:
* **Business Context:** Churn is moderately imbalanced (e.g., 5% to 15% churn).
* **Cost of Errors:**
  * **False Negative (FN):** Failing to identify a customer who is about to churn. This leads to losing the customer's lifetime value (LTV), which is costly.
  * **False Positive (FP):** Predicting a customer will churn when they won't, leading to offering them unnecessary incentives (discounts, free months). This represents wasted retention campaign budget.
* **Metric Choice:** We need a balance. **F1-Score** is the harmonic mean of Precision and Recall. It ensures we don't spam non-churning customers with expensive offers (high Precision) while still capturing most of the customers who are leaving (high Recall).
* **Beta Tuning:** If the retention offer is cheap (e.g. email follow-up), we prioritize Recall using **F2-Score** (β=2). If the offer is expensive (e.g. high discount), we prioritize Precision using **F0.5-Score** (β=0.5).

## Scenario 3: Medical Diagnosis (Rare Disease)

### Metric Selected: Recall (Sensitivity)
* **Primary Metric:** Recall
* **Secondary Metric:** Precision (monitored to control alarm fatigue)

### Justification:
* **Business/Clinical Context:** Detecting a life-threatening or severe disease where early detection is critical.
* **Cost of Errors:**
  * **False Negative (FN):** Missing a sick patient. This is catastrophic as the patient goes untreated, which can lead to severe complications or death. The cost of an FN is extremely high.
  * **False Positive (FP):** Flagging a healthy patient as sick. This leads to temporary anxiety and requires secondary screening (e.g., a follow-up test or biopsy) to confirm. The cost of an FP is acceptable.
* **Metric Choice:** We prioritize **Recall** to be as close to 100% as possible. We accept a lower Precision to guarantee that we do not miss any positive cases. We monitor Precision to ensure healthcare resources are not overwhelmed by false alarms.

## Scenario 4: Dynamic Pricing

### Metric Selected: RMSE (Root Mean Squared Error)
* **Primary Metric:** RMSE
* **Secondary Metric:** MAE (Mean Absolute Error) for baseline comparison

### Justification:
* **Business Context:** Continuous price prediction where small pricing errors are expected but large errors are damaging.
* **Cost of Errors:**
  * **Small errors:** Easily absorbed by market dynamics.
  * **Large errors:** Selling way below cost (severe revenue loss) or selling way above market price (loss of conversion and customer trust).
* **Metric Choice:** **RMSE** squares the errors before averaging them: `sqrt(avg(error^2))`. This quadratic penalty ensures that large prediction errors are penalized much more heavily than small ones. This forces the optimization algorithm to avoid extreme pricing mistakes.

## Edge Cases and Error Cases Analysis

We will explore two key production edge/error cases:

1. **Edge Case 1: Division by Zero in Classification Metrics**
   * When a classifier predicts the majority class 100% of the time, `TP + FP = 0`. Calculating Precision (`TP / (TP + FP)`) results in a division-by-zero.
   * We demonstrate how scikit-learn handles this using the `zero_division` parameter.

2. **Edge Case 2: Extreme Outlier Sensitivity in Regression**
   * We compare how a single extreme outlier shifts MAE vs. RMSE, and how to detect it using their ratio.

In [1]:
# Edge Case 1: Zero Division in Classification Metrics
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

# Scenario: Extremely imbalanced target, model predicts all 0s (majority class)
y_true = np.array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0])
y_pred = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])  # All 0s predicted

print("--- Edge Case 1: All-Negative Predictions ---")
# Standard precision calculation fails with zero-division error or warning
# We use zero_division=0 to return 0.0 instead of throwing an error
precision_default = precision_score(y_true, y_pred, zero_division=0)
recall_default = recall_score(y_true, y_pred, zero_division=0)
f1_default = f1_score(y_true, y_pred, zero_division=0)

print(f"Precision (with zero_division=0): {precision_default:.4f}")
print(f"Recall:                           {recall_default:.4f}")
print(f"F1-Score:                         {f1_default:.4f}")

--- Edge Case 1: All-Negative Predictions ---
Precision (with zero_division=0): 0.0000
Recall:                           0.0000
F1-Score:                         0.0000


In [2]:
# Edge Case 2: Outlier Sensitivity in Regression (MAE vs. RMSE)
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Base predictions and ground truths with small errors
y_true_base = np.array([10.0, 15.0, 20.0, 25.0, 30.0])
y_pred_base = np.array([11.0, 14.0, 21.0, 24.0, 31.0])

mae_base = mean_absolute_error(y_true_base, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_true_base, y_pred_base))

print("--- Edge Case 2: Regression Outlier Sensitivity ---")
print(f"Baseline (No Outliers) - MAE: {mae_base:.4f}, RMSE: {rmse_base:.4f}")

# Introduce a single severe outlier at the end index
y_true_outlier = np.array([10.0, 15.0, 20.0, 25.0, 30.0])
y_pred_outlier = np.array([11.0, 14.0, 21.0, 24.0, 100.0])  # Pred: 100 vs True: 30

mae_outlier = mean_absolute_error(y_true_outlier, y_pred_outlier)
rmse_outlier = np.sqrt(mean_squared_error(y_true_outlier, y_pred_outlier))

print(f"With 1 Outlier         - MAE: {mae_outlier:.4f}, RMSE: {rmse_outlier:.4f}")
print(f"MAE increase:  {mae_outlier - mae_base:.4f} (linear)")
print(f"RMSE increase: {rmse_outlier - rmse_base:.4f} (quadratic)")

--- Edge Case 2: Regression Outlier Sensitivity ---
Baseline (No Outliers) - MAE: 1.0000, RMSE: 1.0000
With 1 Outlier         - MAE: 14.8000, RMSE: 31.3177
MAE increase:  13.8000 (linear)
RMSE increase: 30.3177 (quadratic)


## Reflection and Quality Checks

* **Quality Check 1:** Handled Division by Zero error case in classification metrics using scikit-learn's `zero_division` parameter.
* **Quality Check 2:** Demonstrated how a single outlier heavily skews RMSE compared to MAE in continuous regression tasks.
* **Organized & Free of Junk Code:** Verified that there are no unused imports, dead comments, or custom CSS styling blocks.